# TRM historical data audit

This notebook reports quality evidence for the normalized file. It does not impute, deduplicate, or remove records. Generic Monday-Friday gaps are indicators only: Colombian holidays and source publication rules are not inferred.

In [1]:
from pathlib import Path
import pandas as pd
from trm_forecast.data.audit import audit_trm

DATA_PATH = next(path / "data/interim/trm_normalized.csv" for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/interim/trm_normalized.csv").exists())
data = pd.read_csv(DATA_PATH)
report = audit_trm(data)
data.head()

,date,trm,source,retrieved_at
0,2008-01-03,2012.82,Datos Abiertos Colombia: ceyp-9c7c,2026-10-20 02:00:56.369189
1,2008-01-04,2013.27,Datos Abiertos Colombia: ceyp-9c7c,2026-10-20 02:00:56.369189
2,2008-01-05,2013.98,Datos Abiertos Colombia: ceyp-9c7c,2026-10-20 02:00:56.369189
3,2008-01-09,2000.91,Datos Abiertos Colombia: ceyp-9c7c,2026-10-20 02:00:56.369189
4,2008-01-10,2004.70,Datos Abiertos Colombia: ceyp-9c7c,2026-10-20 02:00:56.369189


In [2]:
summary = pd.DataFrame([{key: value for key, value in report.to_dict().items() if not isinstance(value, (tuple, list))}])
summary.T.rename(columns={0: "value"})

,value
input_rows,4456
observation_count,4456
earliest_date,2008-01-03
latest_date,2026-10-20
unique_date_count,4456
duplicated_date_count,0
rows_in_duplicated_dates,0
missing_date_count,0
invalid_date_count,0
missing_value_count,0


In [3]:
pd.DataFrame(report.calendar_day_gaps)

,start,end,calendar_days,business_days_monday_friday
0,2008-01-03,2008-01-04,1,1
1,2008-01-04,2008-01-05,1,1
2,2008-01-05,2008-01-09,4,2
3,2008-01-09,2008-01-10,1,1
4,2008-01-10,2008-01-11,1,1
...,...,...,...,...
4450,2026-10-10,2026-10-14,4,2
4451,2026-10-14,2026-10-15,1,1
4452,2026-10-15,2026-10-16,1,1
4453,2026-10-16,2026-10-17,1,1


In [4]:
pd.DataFrame(report.candidate_missing_periods)

,start,end,calendar_days,business_days_monday_friday
0,2008-01-05,2008-01-09,4,2
1,2008-01-12,2008-01-15,3,1
2,2008-01-19,2008-01-23,4,2
3,2008-01-26,2008-01-29,3,1
4,2008-02-02,2008-02-05,3,1
...,...,...,...,...
1054,2026-09-19,2026-09-22,3,1
1055,2026-09-26,2026-09-29,3,1
1056,2026-10-03,2026-10-06,3,1
1057,2026-10-10,2026-10-14,4,2


## Interpretation

A candidate period is a pair of consecutive valid observations separated by more than one calendar day. It is not labelled a missing observation: weekends, Colombian public holidays, TRM validity rules, and source outages require external evidence.